# CeNN V5 — Fidelity-first GQA head rescue on Qwen3.5-0.8B

**Goal:** learn whether a **selective hybrid of native attention and CeNN memory** can cross the fidelity gate where V4's memory graph did not. This notebook loads the reusable `tinycenn_lm.cenn_head_rescue_v5` module from the GitHub repository; it does not embed or duplicate the neural layer.

The V4 experiment archived 2026-10-16 found cosine 0.8798 for Delta+Local32, 0.8846 for a learned graph and 0.8863 for a shuffled graph, with **no accepted candidate** (held-out student speedup 0.9807×). Accordingly V5 keeps original Qwen Q/K/V, gate, RoPE, output projection and normalization; it tests increasing CeNN feature/window capacity and **retaining exact SDPA for a complete pretrained GQA head group**, while computing CeNN only on the remaining group. **Exact-all is an oracle/parity control, never a deployment candidate.**

The experiment is **text-only, no cache, no padding** and uses the original Qwen model as the teacher. **No GPU speedup or quality pass is assumed.**


## Research design

- Qwen3.5-0.8B full-attention layer **11** first; optionally test layers 15/19.
- Unchanged frozen pretrained QKV + output gate/projection, original decoder MLP/residual.
- Train seven candidate replacements (three all-CeNN capacity controls, two single native-group rescues, one graph-vs-delta control). Compare to untouched Qwen plus an exact-all functional oracle.
- Fixed WikiText2 **train/validation/test** splits; candidate selection uses *validation only*. Load test only after selection.
- All-token next-token NLL / top-1 agreement; local teacher cosine / NMSE; alternating paired native/student prefill timing and peak allocation.
- Quality gates: cosine ≥ 0.95, teacher agreement ≥ 0.98, ΔNLL ≤ 0.10; final latency gain > 1.05× to claim an accepted speedup.

**Methodological caveat:** head-group rescue explicitly preserves a portion of quadratic attention and may be slower at short contexts. It is a fidelity–compute trade-off, not a novel architectural claim on its own.


In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"


In [ ]:
import os,sys,subprocess,time,gc,json,hashlib,random,math,statistics,copy
from pathlib import Path
from contextlib import contextmanager, nullcontext
from dataclasses import asdict
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Load V5 from its canonical repository source, not from an embedded copy.
REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','main',
                    'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
if str(REPO/'src') not in sys.path:sys.path.insert(0,str(REPO/'src'))
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3
from tinycenn_lm.cenn_memorygraph_v4 import CeNNMemoryGraphV4
from tinycenn_lm.cenn_head_rescue_v5 import Qwen35HeadRescueV5
from transformers import AutoTokenizer,AutoConfig,Qwen3_5ForCausalLM
from huggingface_hub import model_info

GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert torch.cuda.is_available(),'A CUDA GPU runtime is required for the full Qwen experiment.'
DEVICE=torch.device('cuda');DTYPE=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
print('Git SHA',GIT_SHA,'CUDA',torch.cuda.get_device_name(0),'dtype',DTYPE)
subprocess.run([sys.executable,'-m','pytest','-q',str(REPO/'tests/test_cenn_head_rescue_v5.py')],check=True)


## 1. Configuration

Defaults target a **screening run**. `PROFILE='research'` trains longer and uses more data. For a reliable scientific result, run multiple seeds and sequence lengths; a single quick profile does not demonstrate novelty or robust speed.


In [ ]:
# @title Research settings
MODEL_ID='Qwen/Qwen3.5-0.8B'
PROFILE='quick'  # @param ['quick','research']
TARGET_LAYERS=[11]  # Optional: [11,15,19]; only native full-attention layers
SEQ_LEN=256 if PROFILE=='quick' else 512
TRAIN_BLOCKS=48 if PROFILE=='quick' else 128
DEV_BLOCKS=24 if PROFILE=='quick' else 64
TEST_BLOCKS=24 if PROFILE=='quick' else 64
TRAIN_STEPS=140 if PROFILE=='quick' else 600
BATCH_SIZE=2
CHUNK_SIZE=32
LR=2e-3
SEED=2041

# exact_groups indexes into pretrained KV groups. Qwen-0.8B has 2 KV groups,
# each with 4 query heads. Exact rescue computes native SDPA for those heads;
# CeNN is instantiated ONLY for the non-exact groups.
CANDIDATES={
 'delta_f32_w32':dict(kind='delta',f=32,w=32,exact=()),
 'delta_f64_w64':dict(kind='delta',f=64,w=64,exact=()),
 'delta_f96_w64':dict(kind='delta',f=96,w=64,exact=()),
 'rescue_g0_delta_f64_w64':dict(kind='delta',f=64,w=64,exact=(0,)),
 'rescue_g1_delta_f64_w64':dict(kind='delta',f=64,w=64,exact=(1,)),
 'rescue_g0_graph_f64_w64':dict(kind='graph',f=64,w=64,exact=(0,)),
 'rescue_g1_graph_f64_w64':dict(kind='graph',f=64,w=64,exact=(1,)),
}
RESUME_SWEEP=True  # @param {type:'boolean'}
OUT=Path('/content/qwen35_cenn_head_rescue_v5')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('Profile:',PROFILE,'steps',TRAIN_STEPS,'sequence',SEQ_LEN,'candidates',list(CANDIDATES))


## 2. Load pinned pretrained text model

We fail explicitly if HF changes the expected Qwen3.5-0.8B architecture. No weights are ever overwritten on Hugging Face.


In [ ]:
REVISION=model_info(MODEL_ID).sha
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
assert cfg.model_type=='qwen3_5_text',cfg.model_type
assert cfg.num_hidden_layers==24 and cfg.hidden_size==1024,(cfg.num_hidden_layers,cfg.hidden_size)
FULL=[i for i,t in enumerate(cfg.layer_types) if t=='full_attention']
assert FULL==[3,7,11,15,19,23],FULL
assert all(i in FULL for i in TARGET_LAYERS),(TARGET_LAYERS,FULL)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,dtype=DTYPE,
    attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False);model.config.use_cache=False
LAYERS=model.model.layers
for i in TARGET_LAYERS:
    native=LAYERS[i].self_attn
    print('layer',i,'GQA groups:',native.k_proj.out_features//native.head_dim,
          'query heads:',native.q_proj.out_features//(2*native.head_dim),
          'head_dim:',native.head_dim)
print('Model pinned to',REVISION)


## 3. Exact-attention parity oracle

This validates the V5 wrapper before training: rescuing **all** heads must reproduce native attention. A mismatch is a functional error, not an accuracy trade-off. No oracle results enter the candidate ranking.


In [ ]:
@contextmanager
def graft(idx,adapter):
    original=LAYERS[idx].self_attn
    assert adapter.original is original
    LAYERS[idx].self_attn=adapter
    try:yield
    finally:LAYERS[idx].self_attn=original

sample_ids=tokenizer('Neural memory architectures can preserve information across context.',
                     return_tensors='pt',add_special_tokens=False).input_ids.to(DEVICE)
for idx in TARGET_LAYERS:
    original=LAYERS[idx].self_attn
    groups=int(original.k_proj.out_features//original.head_dim)
    oracle=Qwen35HeadRescueV5(original,None,exact_groups=range(groups)).to(DEVICE).eval()
    with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
        ref=model(input_ids=sample_ids,use_cache=False,logits_to_keep=0).logits.float()
        with graft(idx,oracle):
            check=model(input_ids=sample_ids,use_cache=False,logits_to_keep=0).logits.float()
    max_abs=float((ref-check).abs().max())
    top_agree=float((ref.argmax(-1)==check.argmax(-1)).float().mean())
    print(f'Layer {idx} exact oracle: logits max abs {max_abs:.6f}, top1 match {top_agree:.4f}')
    assert max_abs<0.05 and top_agree==1.0,'Qwen native attention parity failed; abort before training'
    del oracle


## 4. Create disjoint WikiText2 data partitions

Fixed unpadded sequences. Train and validation are used for distillation and selection. The held-out test split is **not loaded** until the final audit cell.


In [ ]:
from datasets import load_dataset

def token_blocks(split,count,length):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    tokens=[]
    for row in ds:
        text=row['text'].strip()
        if not text:continue
        tokens.extend(tokenizer.encode(text,add_special_tokens=False))
        tokens.append(tokenizer.eos_token_id)
        if len(tokens)>=(count+1)*length:break
    if len(tokens)<count*length:
        raise RuntimeError(f'{split}: collected {len(tokens)}, need {count*length}')
    return torch.tensor(tokens[:count*length],dtype=torch.long).reshape(count,length)

TRAIN=token_blocks('train',TRAIN_BLOCKS,SEQ_LEN)
DEV=token_blocks('validation',DEV_BLOCKS,SEQ_LEN)
PARTITION_HASHES={label:hashlib.sha256(t.numpy().tobytes()).hexdigest()
                  for label,t in [('train',TRAIN),('validation',DEV)]}
assert len(set(PARTITION_HASHES.values()))==2
print('train/dev',TRAIN.shape,DEV.shape,'sha256',PARTITION_HASHES)


## 5. Capture native Qwen teacher activations

Distill only the **full-attention mixer output**; original decoder normalization, residual and MLP remain unchanged. Cache unpadded mixer input, mixer target, and original position embeddings on CPU.


In [ ]:
TEACHER={i:{} for i in TARGET_LAYERS}
@torch.inference_mode()
def native_forward(ids,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=ids.to(DEVICE),use_cache=False,logits_to_keep=keep)

def capture(split,ids):
    rows={i:{'x':[],'y':[],'cos':[],'sin':[]} for i in TARGET_LAYERS}
    handles=[]
    for idx in TARGET_LAYERS:
        def hook(module,args,kwargs,out,j=idx):
            x=kwargs.get('hidden_states',args[0] if args else None)
            assert x is not None
            y=out[0] if isinstance(out,tuple) else out
            c,s=kwargs['position_embeddings']
            rows[j]['x'].append(x.detach().cpu().to(torch.float16))
            rows[j]['y'].append(y.detach().cpu().to(torch.float16))
            rows[j]['cos'].append(c.detach().cpu().to(torch.float32))
            rows[j]['sin'].append(s.detach().cpu().to(torch.float32))
        handles.append(LAYERS[idx].self_attn.register_forward_hook(hook,with_kwargs=True))
    try:
        for i in range(0,len(ids),BATCH_SIZE):native_forward(ids[i:i+BATCH_SIZE])
    finally:
        for h in handles:h.remove()
    for idx in TARGET_LAYERS:
        TEACHER[idx][split]={name:torch.cat(values,0).contiguous()
                             for name,values in rows[idx].items()}
        print('capture',split,idx,'x/y',TEACHER[idx][split]['x'].shape)

capture('train',TRAIN);capture('dev',DEV)
torch.cuda.empty_cache()


## 6. Build CeNN-only and selective-rescue candidates

The **approximate core size is proportional to the number of non-rescued KV groups**. Exact heads are computed by native SDPA only; no redundant recurrent work is executed for rescued heads.


In [ ]:
def make_candidate(idx,name):
    choice=CANDIDATES[name]
    original=LAYERS[idx].self_attn
    head_dim=int(original.head_dim)
    heads=int(original.q_proj.out_features//(2*head_dim))
    kvheads=int(original.k_proj.out_features//head_dim)
    groups=heads//kvheads
    exact=tuple(choice['exact'])
    approximate=kvheads-len(exact)
    common=dict(num_heads=approximate*groups,num_kv_heads=approximate,
                head_dim=head_dim,feature_dim=choice['f'],chunk_size=CHUNK_SIZE,
                local_window=choice['w'])
    if not approximate:core=None
    elif choice['kind']=='delta':core=CeNNDeltaCellV3(**common,mode='none')
    elif choice['kind']=='graph':
        core=CeNNMemoryGraphV4(**common,graph_mode='learned',top_k=4,
                                edge_dim=12,content_adaptive=True,query_router=True)
    else:raise ValueError(choice)
    if core is not None:core=core.to(DEVICE)
    adapter=Qwen35HeadRescueV5(original,core,exact_groups=exact)
    assert all(not p.requires_grad for p in original.parameters())
    return adapter,dict(**choice,approximate_kv_groups=approximate,
                        exact_heads=adapter.num_exact_heads,
                        approximate_heads=adapter.num_approx_heads,
                        trainable_params=sum(p.numel() for p in core.parameters()) if core else 0)

def batch_capture(idx,split,indices):
    d=TEACHER[idx][split]
    return (d['x'][indices].to(DEVICE,dtype=DTYPE),
            d['y'][indices].to(DEVICE,dtype=DTYPE),
            (d['cos'][indices].to(DEVICE),d['sin'][indices].to(DEVICE)))

def local_loss(pred,target):
    pp,tt=pred.float(),target.float()
    mse=(pp-tt).square().mean()/tt.square().mean().clamp_min(1e-8)
    cosine=F.cosine_similarity(pp,tt,dim=-1).mean()
    return mse + 0.2*(1-cosine),mse,cosine

@torch.no_grad()
def validation_local(adapter,idx):
    adapter.eval();err=energy=cos_sum=0.;n=0
    for j in range(0,DEV_BLOCKS,BATCH_SIZE):
        subset=torch.arange(j,min(j+BATCH_SIZE,DEV_BLOCKS))
        x,y,rope=batch_capture(idx,'dev',subset)
        with torch.autocast('cuda',dtype=DTYPE):pred,_=adapter(x,position_embeddings=rope)
        e,ref=pred.float(),y.float()
        err+=(e-ref).square().sum().item();energy+=ref.square().sum().item()
        cos_sum+=F.cosine_similarity(e,ref,dim=-1).sum().item()
        n+=e.shape[0]*e.shape[1]
    return dict(cosine=cos_sum/n,nmse=err/max(energy,1e-8))


## 7. Train replacement cores with teacher-output distillation

Every candidate is independently initialized and uses the same training examples and number of update steps. The pretrained Qwen parameters remain frozen. Validation chooses the **best local checkpoint**, while the test remains unread.


In [ ]:
def fit_candidate(idx,name,steps=TRAIN_STEPS):
    torch.manual_seed(SEED+idx*19)
    adapter,config=make_candidate(idx,name)
    assert adapter.core is not None
    adapter.train()
    opt=torch.optim.AdamW(adapter.core.parameters(),lr=LR,weight_decay=0.01)
    schedule=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=steps,eta_min=LR*0.08)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    best=float('inf');best_sd=None;best_local=None
    for step in range(1,steps+1):
        inds=torch.randint(0,TRAIN_BLOCKS,(BATCH_SIZE,))
        x,y,rope=batch_capture(idx,'train',inds)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred,_=adapter(x,position_embeddings=rope)
            loss,_,_=local_loss(pred,y)
        if not bool(torch.isfinite(loss)):raise FloatingPointError((idx,name,step,loss.item()))
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(adapter.core.parameters(),1.0)
        scaler.step(opt);scaler.update();schedule.step()
        if step==1 or step%max(1,steps//6)==0 or step==steps:
            local=validation_local(adapter,idx)
            score=local['nmse']+0.2*(1-local['cosine'])
            if score<best:
                best=score;best_local=local
                best_sd={key:value.detach().cpu().clone() for key,value in adapter.core.state_dict().items()}
            print(f'L{idx} {name} {step}/{steps}: loss {loss.item():.4f}, '
                  f'val cosine {local["cosine"]:.4f}, NMSE {local["nmse"]:.4f}',flush=True)
            adapter.train()
    adapter.core.load_state_dict(best_sd);adapter.eval()
    return adapter,config,best_local,best_sd


## 8. Model-level validation with all next-token positions

The next-token score uses positions 0…T−2 of each unpadded block, not a small handpicked subset. Baseline predictions are cached as integer IDs, avoiding a large teacher-logit copy.


In [ ]:
@torch.inference_mode()
def evaluate_logits(ids,teacher_top1=None):
    n=0;nll=0.;gold=0;agree=0;teacher=[]
    for begin in range(0,len(ids),BATCH_SIZE):
        chunk=ids[begin:begin+BATCH_SIZE]
        pos=torch.arange(chunk.shape[1]-1,device=DEVICE)
        with torch.autocast('cuda',dtype=DTYPE):
            logits=model(input_ids=chunk.to(DEVICE),use_cache=False,
                         logits_to_keep=pos).logits.float()
        labels=chunk[:,1:].to(DEVICE)
        assert logits.shape[:2]==labels.shape
        logp=F.log_softmax(logits,dim=-1)
        nll-=logp.gather(-1,labels.unsqueeze(-1)).sum().item()
        pred=logits.argmax(-1)
        gold+=(pred==labels).sum().item();n+=labels.numel()
        if teacher_top1 is None:teacher.append(pred.cpu().to(torch.int32))
        else:
            top=teacher_top1[begin//BATCH_SIZE].to(DEVICE)
            agree+=(pred==top).sum().item()
        del logits,logp
    result=dict(nll=nll/n,next_token_accuracy=gold/n,decisions=n)
    if teacher_top1 is not None:result['teacher_agreement']=agree/n
    return result,teacher

@torch.inference_mode()
def paired_prefill(idx,adapter,ids,repeats=12,warmups=3):
    # Both models see exactly the same sample and logits_to_keep=1.
    sample=ids[:min(BATCH_SIZE,len(ids))].to(DEVICE)
    native_times=[];student_times=[]
    def once(student):
        with graft(idx,adapter) if student else nullcontext():
            a=torch.cuda.Event(enable_timing=True);b=torch.cuda.Event(enable_timing=True)
            a.record()
            with torch.autocast('cuda',dtype=DTYPE):
                result=model(input_ids=sample,use_cache=False,logits_to_keep=1).logits
            b.record();torch.cuda.synchronize()
            assert bool(torch.isfinite(result).all())
            return a.elapsed_time(b)
    torch.cuda.synchronize()
    for i in range(warmups+repeats):
        if i%2==0:nv=once(False);sv=once(True)
        else:sv=once(True);nv=once(False)
        if i>=warmups:native_times.append(nv);student_times.append(sv)
    native_ms=float(np.median(native_times));student_ms=float(np.median(student_times))
    return dict(native_ms=native_ms,student_ms=student_ms,
                speedup=native_ms/student_ms,
                native_iqr_ms=float(np.subtract(*np.percentile(native_times,[75,25]))),
                student_iqr_ms=float(np.subtract(*np.percentile(student_times,[75,25]))),
                repeats=repeats)

NATIVE_DEV,NATIVE_DEV_TOP1=evaluate_logits(DEV)
print('Native validation:',NATIVE_DEV)


## 9. Candidate sweep

Completed candidates can be reused if their checkpoint and run fingerprint match. Each result includes local representation fit, full-model validation NLL/agreement, and alternating GPU latency. **Never select on held-out test performance**.


In [ ]:
# Guard against accidental reuse after changes to training, model or code.
RUN_SPEC=dict(model_id=MODEL_ID,model_revision=REVISION,repo_sha=GIT_SHA,
              profile=PROFILE,layers=TARGET_LAYERS,candidates=CANDIDATES,
              seq_len=SEQ_LEN,train_blocks=TRAIN_BLOCKS,dev_blocks=DEV_BLOCKS,
              seed=SEED,steps=TRAIN_STEPS,lr=LR,chunk_size=CHUNK_SIZE)
RUN_FINGERPRINT=hashlib.sha256(json.dumps(RUN_SPEC,sort_keys=True).encode()).hexdigest()
RESULTS_FILE=OUT/'validation_candidates.csv'
previous={}
if RESUME_SWEEP and RESULTS_FILE.exists():
    earlier=pd.read_csv(RESULTS_FILE)
    if 'run_fingerprint' in earlier.columns:
        for row in earlier.to_dict('records'):
            if str(row.get('run_fingerprint'))==RUN_FINGERPRINT:
                previous[(int(row['layer']),str(row['variant']))]=row
REPORT=[]
for idx in TARGET_LAYERS:
    for name in CANDIDATES:
        print(f'\n===== L{idx} {name} =====',flush=True)
        checkpoint=OUT/'checkpoints'/f'v5_L{idx}_{name}.pt'
        key=(int(idx),str(name))
        if key in previous and checkpoint.exists():
            saved=torch.load(checkpoint,map_location='cpu',weights_only=True)
            if saved.get('run_fingerprint')==RUN_FINGERPRINT:
                REPORT.append(previous[key])
                print('Resumed completed candidate:',idx,name,flush=True)
                continue
        student,conf,local,weights=fit_candidate(idx,name)
        with graft(idx,student):
            dev,_=evaluate_logits(DEV,teacher_top1=NATIVE_DEV_TOP1)
        timing=paired_prefill(idx,student,DEV,repeats=6,warmups=2)
        record=dict(layer=int(idx),variant=name,**conf,steps=TRAIN_STEPS,
                    run_fingerprint=RUN_FINGERPRINT,
                    local_dev_cosine=local['cosine'],local_dev_nmse=local['nmse'],
                    dev_nll=dev['nll'],dev_delta_nll=dev['nll']-NATIVE_DEV['nll'],
                    dev_agreement=dev['teacher_agreement'],decisions=dev['decisions'],
                    paired_speedup=timing['speedup'],native_ms=timing['native_ms'],
                    student_ms=timing['student_ms'])
        REPORT.append(record)
        torch.save({'core_state':weights,'config':conf,'layer':idx,'variant':name,
                    'model_revision':REVISION,'repo_sha':GIT_SHA,
                    'run_fingerprint':RUN_FINGERPRINT,
                    'cache_support':False},OUT/'checkpoints'/f'v5_L{idx}_{name}.pt')
        pd.DataFrame(REPORT).to_csv(OUT/'validation_candidates.csv',index=False)
        print('RESULT',json.dumps(record,indent=2),flush=True)
        del student,weights;gc.collect();torch.cuda.empty_cache()
print('Trained candidates:',len(REPORT))


## 10. Strict validation ranking

We prefer **quality** over a potentially misleading microbenchmark. If no candidate passes, the selected candidate is diagnostic only; the original Qwen checkpoint remains the recommended deployment choice.


In [ ]:
MIN_COSINE=.95
MIN_AGREEMENT=.98
MAX_DELTA_NLL=.10
ranked=pd.DataFrame(REPORT)
ranked['quality_gate']=((ranked.local_dev_cosine>=MIN_COSINE)&
                        (ranked.dev_agreement>=MIN_AGREEMENT)&
                        (ranked.dev_delta_nll<=MAX_DELTA_NLL))
ranked=ranked.sort_values(['quality_gate','dev_delta_nll','local_dev_nmse'],
                          ascending=[False,True,True]).reset_index(drop=True)
display(ranked[['layer','variant','exact_heads','approximate_heads',
                'local_dev_cosine','local_dev_nmse','dev_delta_nll',
                'dev_agreement','paired_speedup','quality_gate']])
qualified=ranked[ranked.quality_gate]
if len(qualified):
    winner=qualified.sort_values(['paired_speedup','dev_delta_nll'],ascending=[False,True]).iloc[0]
    quality_accepted=True
else:
    winner=ranked.iloc[0];quality_accepted=False
print('Selected for audit:',winner.layer,winner.variant,'validation quality accepted:',quality_accepted)
ranked.to_csv(OUT/'validation_ranked.csv',index=False)


## 11. One held-out WikiText2 test audit

Test is **loaded after validation selection**. The test run is not repeated to tune hyperparameters. Pair native/student timing with alternating order, measure the same prefill workload and record a performance gate separately from fidelity.


In [ ]:
WIN_LAYER=int(winner.layer);WIN_NAME=str(winner.variant)
payload=torch.load(OUT/'checkpoints'/f'v5_L{WIN_LAYER}_{WIN_NAME}.pt',
                   map_location='cpu',weights_only=True)
student,restored_conf=make_candidate(WIN_LAYER,WIN_NAME)
assert payload.get('run_fingerprint')==RUN_FINGERPRINT,'Checkpoint/settings mismatch'
student.core.load_state_dict(payload['core_state']);student.eval()

# Test was not consulted while designing or selecting candidates.
TEST=token_blocks('test',TEST_BLOCKS,SEQ_LEN)
PARTITION_HASHES['test']=hashlib.sha256(TEST.numpy().tobytes()).hexdigest()
assert len(set(PARTITION_HASHES.values()))==3
TEST_NATIVE,TEST_NATIVE_TOP1=evaluate_logits(TEST)
with graft(WIN_LAYER,student):
    TEST_STUDENT,_=evaluate_logits(TEST,teacher_top1=TEST_NATIVE_TOP1)
TEST_DELTA=TEST_STUDENT['nll']-TEST_NATIVE['nll']
TEST_TIMING=paired_prefill(WIN_LAYER,student,TEST,repeats=18,warmups=5)
FINAL_PASS=bool(quality_accepted and TEST_DELTA<=MAX_DELTA_NLL and
                TEST_STUDENT['teacher_agreement']>=MIN_AGREEMENT and
                TEST_TIMING['speedup']>1.05)
AUDIT=dict(architecture='CeNN V5 selective head rescue',model_id=MODEL_ID,
           model_sha=REVISION,repo_sha=GIT_SHA,profile=PROFILE,seed=SEED,
           selected=dict(layer=WIN_LAYER,variant=WIN_NAME),
           validation_accepted=quality_accepted,final_pass=FINAL_PASS,
           quality_thresholds=dict(min_cosine=MIN_COSINE,min_agreement=MIN_AGREEMENT,
                                   max_delta_nll=MAX_DELTA_NLL,min_speedup=1.05),
           native_test=TEST_NATIVE,student_test=TEST_STUDENT,
           test_delta_nll=TEST_DELTA,paired_test_timing=TEST_TIMING,
           all_candidates=REPORT,split_hashes=PARTITION_HASHES,
           limitations='No KV-cache decoding; no padded batches/multimodal tests; native SDPA anchors '
                       'retain quadratic attention; no fused CeNN kernel; quick profile single-seed screening.')
(OUT/'final_audit.json').write_text(json.dumps(AUDIT,indent=2,allow_nan=False))
print(json.dumps({k:AUDIT[k] for k in ['selected','validation_accepted','final_pass',
             'native_test','student_test','test_delta_nll','paired_test_timing']},indent=2))
if not FINAL_PASS:print('NO ACCEPTED SPEED+QUALITY IMPROVEMENT: retain original native Qwen layer')


## 12. Optional GPU length sweep

Screen multiple sequence lengths with identical batch size and implementation. This is prefill only, **not a cached-token latency benchmark**. Longer sequences than the distillation context can exhibit quality regressions.


In [ ]:
# @title Optional length sweep
RUN_LENGTH_SWEEP=False # @param {type:'boolean'}
LENGTH_REPORT=[]
if RUN_LENGTH_SWEEP:
    for length in [128,256,512,1024,2048]:
        try:
            x=token_blocks('validation',4,length)
            r=paired_prefill(WIN_LAYER,student,x,repeats=8,warmups=3)
            LENGTH_REPORT.append(dict(seq_len=length,**r))
            print('length',length,r)
        except torch.cuda.OutOfMemoryError:
            print('OOM at',length,'tokens; skipping further lengths')
            torch.cuda.empty_cache()
            break
    pd.DataFrame(LENGTH_REPORT).to_csv(OUT/'prefill_length_sweep.csv',index=False)
    AUDIT['prefill_length_sweep']=LENGTH_REPORT
    (OUT/'final_audit.json').write_text(json.dumps(AUDIT,indent=2,allow_nan=False))


## 13. Visualization and downloadable results

The ZIP contains validation tables, test audit and adapter-only checkpoints. The pretrained Qwen model is **not** included or modified.


In [ ]:
import shutil
fig,ax=plt.subplots(figsize=(10,5))
f=ranked.sort_values('local_dev_cosine',ascending=True)
ax.barh(f.variant,f.local_dev_cosine)
ax.axvline(MIN_COSINE,linestyle='--',color='gray',label='cosine quality gate')
ax.set(xlim=(0,1),xlabel='Local teacher cosine',title='V5: Precision rescue vs all-CeNN')
ax.legend();fig.tight_layout();fig.savefig(OUT/'validation_cosine.png',dpi=170)
(OUT/'manifest.json').write_text(json.dumps(dict(
 model=MODEL_ID,model_revision=REVISION,source_git_sha=GIT_SHA,
 experiment='CeNN V5 Head Rescue',notebook='Qwen35_08B_CeNN_HeadRescue_V5_Quality_First_Colab.ipynb',
 no_cache=True,architecture_source='tinycenn_lm.cenn_head_rescue_v5',
 profile=PROFILE,variants=list(CANDIDATES),selection='validation only',
 test='WikiText2 test held out until final audit'),indent=2))
shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Archive:',str(OUT)+'.zip')
from IPython.display import display,FileLink
display(FileLink(str(OUT)+'.zip'))


## Interpreting this experiment

A good result must demonstrate (1) quality acceptance at **layer 11**, (2) held-out quality nonregression and (3) repeated end-to-end speed gain at deployment-relevant lengths. The exact-all oracle cannot demonstrate a speedup because it is the unchanged native operation. If group rescue passes quality but is slower, that still identifies the accuracy-sensitive attention head group; optimize CeNN CUDA kernels **after** this decision. 

**V4 finding reproduced in motivation:** learned graph did not beat shuffled graph on the uploaded quick run. The new V5 isolates exact retrieval preservation rather than claiming that additional graph edges are inherently novel. Literature comparison and independent seeds/models are needed before making originality claims.

**Reproducibility:** The notebook uses the repository V5 module, adds a symmetric graph-head control, and resumes only candidates with matching source/model/hyperparameter fingerprints.
